In [79]:
import pandas as pd

In [80]:
df=pd.read_csv("Emotion_Sentiment_DataSet.csv")

In [81]:
df.shape


(160000, 3)

In [82]:
df.keys()

Index(['Unnamed: 0', 'Text', 'Emotion'], dtype='str')

In [83]:
df.count()

Unnamed: 0    160000
Text          159992
Emotion       160000
dtype: int64

In [84]:
df.columns

Index(['Unnamed: 0', 'Text', 'Emotion'], dtype='str')

In [85]:


print(df.head())
print("\nColumns:")
print(df.columns)

print("\nShape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

   Unnamed: 0                                               Text Emotion
0           0  i feel jealous becasue i wanted that kind of l...    love
1           1  i feel like she has taken on the role of a gra...    love
2           2  i feel like im back to the arms of a beloved l...    love
3           3  i am feeling so festive right now and not just...    love
4           4  i don t discuss even my feelings for beloved w...    love

Columns:
Index(['Unnamed: 0', 'Text', 'Emotion'], dtype='str')

Shape:
(160000, 3)

Missing values:
Unnamed: 0    0
Text          8
Emotion       0
dtype: int64


In [86]:
print(df["Emotion"].value_counts())
print("\nNumber of classes:", df["Emotion"].nunique())

Emotion
love          39553
happiness     27175
sadness       17481
Normal        16351
hate          15267
anger         12336
Depression    10333
fun           10075
surprise       6954
worry          4475
Name: count, dtype: int64

Number of classes: 10


In [87]:
from sklearn.utils import resample
min_sample=df["Emotion"].value_counts().min()
min_sample

np.int64(4475)

In [88]:
df_resample = pd.concat(
    [
        group.sample(n=min_sample, random_state=73)
        for _, group in df.groupby("Emotion")
    ],
    ignore_index=True
)

In [89]:
df_resample.columns

Index(['Unnamed: 0', 'Text', 'Emotion'], dtype='str')

In [90]:
print(df_resample["Emotion"].value_counts())

Emotion
Depression    4475
Normal        4475
anger         4475
fun           4475
happiness     4475
hate          4475
love          4475
sadness       4475
surprise      4475
worry         4475
Name: count, dtype: int64


In [91]:
df_resample["Text"] = df_resample["Text"].fillna("").astype(str)

In [92]:
df_resample.head()

,Unnamed: 0,Text,Emotion
0,157631,"I know it will pass, but right now I just feel...",Depression
1,149683,Not a day goes by that I do not break down in ...,Depression
2,153107,I have 11 days until my lease is up. I went an...,Depression
3,150541,I cannot sleep no matter how tired I am always...,Depression
4,157728,"Lately, I have been feeling quite restless and...",Depression


In [93]:
from sklearn.preprocessing import LabelEncoder

In [94]:
label =LabelEncoder()

In [95]:
df_resample["label"]=label.fit_transform(df_resample["Emotion"])

In [96]:
df_resample.head()

,Unnamed: 0,Text,Emotion,label
0,157631,"I know it will pass, but right now I just feel...",Depression,0
1,149683,Not a day goes by that I do not break down in ...,Depression,0
2,153107,I have 11 days until my lease is up. I went an...,Depression,0
3,150541,I cannot sleep no matter how tired I am always...,Depression,0
4,157728,"Lately, I have been feeling quite restless and...",Depression,0


In [97]:
label_mapping = dict(
    zip(
        label.classes_,
        label.transform(label.classes_)
    )
)

print(label_mapping)

{'Depression': np.int64(0), 'Normal': np.int64(1), 'anger': np.int64(2), 'fun': np.int64(3), 'happiness': np.int64(4), 'hate': np.int64(5), 'love': np.int64(6), 'sadness': np.int64(7), 'surprise': np.int64(8), 'worry': np.int64(9)}


In [98]:
from sklearn.model_selection import train_test_split

In [99]:
train_df,temp_df=train_test_split(
    df_resample,
    test_size=0.2,
    random_state=73,
    stratify=df_resample["label"]
    
)

In [100]:
test_df,val_df=train_test_split(
    temp_df,
    test_size=.5,
    random_state=73,
    stratify=temp_df["label"]
    
)

In [101]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (35800, 4)
Validation: (4475, 4)
Test: (4475, 4)


In [102]:
from datasets import Dataset

In [103]:
train_dataset=Dataset.from_pandas(train_df[["Text","label"]])

In [104]:
test_dataset=Dataset.from_pandas(test_df[["Text","label"]])

In [105]:
val_dataset=Dataset.from_pandas(val_df[["Text","label"]])

In [106]:
print(train_dataset)

Dataset({
    features: ['Text', 'label', '__index_level_0__'],
    num_rows: 35800
})


In [107]:
from transformers import AutoTokenizer

In [108]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [109]:
example = "I am extremely happy today!"

tokens = tokenizer(example)

print(tokens)

{'input_ids': [101, 1045, 2572, 5186, 3407, 2651, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [110]:
def tokenize_function(examples):
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [111]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/35800 [00:00<?, ? examples/s]

In [113]:
tokenized_test =test_dataset.map(tokenize_function,batched=True)

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]

In [114]:
tokenized_val =val_dataset.map(tokenize_function,batched=True)

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]

In [115]:
print(tokenized_train)

Dataset({
    features: ['Text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 35800
})


In [126]:
print(tokenized_test[0])

{'Text': 'im still feeling like its a girl in there but i will not be at all surprised if it is a boy because my mind is messing with me and everyone keeps telling me they think its a boy', 'label': 8, '__index_level_0__': 36947, 'input_ids': [101, 10047, 2145, 3110, 2066, 2049, 1037, 2611, 1999, 2045, 2021, 1045, 2097, 2025, 2022, 2012, 2035, 4527, 2065, 2009, 2003, 1037, 2879, 2138, 2026, 2568, 2003, 22308, 2007, 2033, 1998, 3071, 7906, 4129, 2033, 2027, 2228, 2049, 1037, 2879, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [127]:
print(tokenized_train)
print(tokenized_train[0])

Dataset({
    features: ['Text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 35800
})
{'Text': 'i feel the urge to have cute and painted toe nails but feel like i cant because my big toenail is still growing in and look super short and funny right now', 'label': 3, '__index_level_0__': 14001, 'input_ids': [101, 1045, 2514, 1996, 9075, 2000, 2031, 10140, 1998, 4993, 11756, 10063, 2021, 2514, 2066, 1045, 2064, 2102, 2138, 2026, 2502, 11756, 25464, 2003, 2145, 3652, 1999, 1998, 2298, 3565, 2460, 1998, 6057, 2157, 2085, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [132]:
from transformers import AutoModelForSequenceClassification

num_labels = 10

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
